# 02 — Chia dữ liệu theo người và khóa split
**Đích:** cùng một người không xuất hiện ở train, validation và inter-test.
Đầu vào: audit đã kiểm. Đầu ra: events/subjects/recordings có split + checksum.
Không mở WAV; không chọn split bằng điểm classifier. Mọi lần chạy tạo thư mục mới,
nhưng cùng audit và seed phải cho cùng danh sách người. Sau freeze không đổi seed để tăng điểm.
Chạy **Restart Kernel → Run All** với kernel DSP. Tên thư mục output cuối dùng làm SPLIT_ID ở notebook 03.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, json, platform, sys, time, uuid
import importlib.metadata as metadata
import numpy as np
from IPython.display import display, Markdown

PROJECT_PATH = ""  # Chỉ điền thư mục gốc nếu kernel mở ngoài dự án.
print("Kernel:", sys.executable, "\nCWD:", Path.cwd())
if PROJECT_PATH and sys.platform != "win32" and ":" in PROJECT_PATH:
    raise ValueError("Đang dùng kernel Linux với đường dẫn Windows. Chọn kernel DSP local.")
candidates = [Path(PROJECT_PATH)] if PROJECT_PATH else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p / 'requirements-dsp.txt').is_file() and (p / 'dsp/notebooks').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Điền PROJECT_PATH là thư mục chứa requirements-dsp.txt; xem README.md")
STARTED = time.perf_counter()

def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fields=None):
    with path.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields or list(rows[0]))
        w.writeheader()
        w.writerows(rows)

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def verify(summary):
    for relative, expected in summary['output_sha256'].items():
        assert sha256(ROOT / relative) == expected, f"Checksum sai: {relative}"

def save_summary(path, summary, outputs):
    summary.update(created_at_utc=datetime.now(timezone.utc).isoformat(),
                   python=sys.version, executable=sys.executable, platform=platform.platform(),
                   versions={n: metadata.version(n) for n in ['numpy','scipy','matplotlib','ipykernel','nbformat']},
                   elapsed_seconds=time.perf_counter()-STARTED,
                   output_sha256={str(p.relative_to(ROOT)).replace('\\','/'): sha256(p) for p in outputs})
    path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print("Đã lưu:", path.relative_to(ROOT))

def snapshot(name, destination):
    import nbformat
    nb = nbformat.read(ROOT / 'dsp/notebooks' / name, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == 'code':
            cell.outputs, cell.execution_count = [], None
    nbformat.write(nb, destination)

## 1. Tham số và đầu vào
20% của **toàn bộ 251 official-training subjects**, kể cả người không có event thuộc task.
Grouped shuffle: sắp ID, hoán vị nhóm bằng NumPy RandomState(seed), lấy ceil(20% × số người)
đầu tiên làm validation. Đây là cách chia nhóm trực tiếp; không cần thêm scikit-learn.
Thử 2026–2035 và dừng ở seed đầu tiên có cả hai lớp. Không thay điều kiện thành cân bằng số event.

In [ ]:
AUDIT_ID = 'audit_20261026T174438Z_a0c82a'
VAL_RATIO = 0.20
SEEDS = range(2026, 2036)
audit_path = ROOT / 'dsp/outputs/reports' / AUDIT_ID / 'audit_summary.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
assert audit['ready_for_split_review'] and audit['error_count'] == audit['review_count'] == 0
verify(audit)
source = ROOT / 'dsp/data/manifests' / AUDIT_ID
events = read_csv(source / 'events.csv')
recordings = read_csv(source / 'recordings.csv')
subjects = sorted({r['subject_id'] for r in recordings if r['official_partition']=='official_train'})
test_subjects = {r['subject_id'] for r in recordings if r['official_partition']=='inter_test'}
assert set(subjects).isdisjoint(test_subjects)
print('Official train:', len(subjects), 'người; inter-test:', len(test_subjects), 'người')

## 2. Chọn seed từ metadata
Một người có nhiều event vẫn chỉ được bốc thăm một lần. `included` ở bước này là annotation eligibility;
chưa phải reviewed synthesis anchor. Intra-test không tham gia core dù có cùng người với train.

In [ ]:
def grouped_subjects(ids, seed, ratio):
    shuffled = np.random.RandomState(seed).permutation(sorted(ids))
    n_val = int(np.ceil(len(ids) * ratio))
    return set(shuffled[n_val:].tolist()), set(shuffled[:n_val].tolist())

attempts = []
eligible = [e for e in events if e['official_partition']=='official_train' and e['included']=='true']
for seed in SEEDS:
    train_subjects, val_subjects = grouped_subjects(subjects, seed, VAL_RATIO)
    train_labels = {e['label'] for e in eligible if e['subject_id'] in train_subjects}
    val_labels = {e['label'] for e in eligible if e['subject_id'] in val_subjects}
    feasible = train_labels == val_labels == {'Normal','Wheeze'}
    attempts.append({'seed':seed, 'both_classes':feasible})
    if feasible:
        selected_seed = seed
        break
else:
    raise RuntimeError('Không seed nào hỗ trợ hai lớp. Dừng và chuẩn bị câu hỏi cho thầy.')
print('Seed đầu tiên hợp lệ:', selected_seed, attempts)

## 3. Gắn split vào từng nguồn
Event ngoài task vẫn được giữ với `included=false`, để notebook crop kiểm tra được context lân cận.
Test chỉ có metadata, không xuất segment hoặc feature tại tuần 1.

In [ ]:
def partition_for(row):
    if row['official_partition'] == 'intra_test_unused': return 'unused'
    if row['official_partition'] == 'inter_test': return 'test'
    return 'train' if row['subject_id'] in train_subjects else 'validation'

for row in events + recordings:
    row['split'] = partition_for(row)
subject_rows = [{'subject_id': s, 'split': split, 'official_partition': official}
                for split, ids, official in [('train',train_subjects,'official_train'),
                                             ('validation',val_subjects,'official_train'),
                                             ('test',test_subjects,'inter_test')]
                for s in sorted(ids)]
counts = []
for part in ['train','validation','test']:
    rr = [r for r in recordings if r['split']==part]
    ee = [e for e in events if e['split']==part]
    inc = [e for e in ee if e['included']=='true']
    counts.append(dict(split=part, subjects=len({r['subject_id'] for r in rr}),
        eligible_subjects=len({e['subject_id'] for e in inc}), recordings=len(rr),
        annotated_events=len(ee), eligible_events=len(inc), excluded_events=len(ee)-len(inc),
        normal=sum(e['label']=='Normal' for e in inc), wheeze=sum(e['label']=='Wheeze' for e in inc),
        wheeze_subjects=len({e['subject_id'] for e in inc if e['label']=='Wheeze'})))
display(counts)

## 4. Kiểm chứng split
Kiểm tra cả giao người lẫn WAV byte-identical giữa core partitions (checksum từ lần tải).
Không đọc lại test audio. Kiểm tra đổi thứ tự đầu vào không thay kết quả để tránh lệ thuộc thứ tự CSV.

In [ ]:
assert train_subjects | val_subjects == set(subjects)
assert not (train_subjects & val_subjects or train_subjects & test_subjects or val_subjects & test_subjects)
assert grouped_subjects(list(reversed(subjects)), selected_seed, VAL_RATIO) == (train_subjects,val_subjects)
assert len({r['subject_id'] for r in subject_rows}) == len(subject_rows)
hash_parts = defaultdict(set)
for r in recordings:
    if r['split'] != 'unused': hash_parts[r['audio_sha256_at_download']].add(r['split'])
assert all(len(parts)==1 for parts in hash_parts.values()), 'WAV trùng vượt partition'
assert all(e['included']=='false' for e in events if e['split']=='unused')
assert all(c['normal'] > 0 and c['wheeze'] > 0 for c in counts)
print('PASS: giao người = 0; không WAV trùng vượt core partitions; hai lớp hiện diện.')

## 5. Lưu split và dấu vết
`split_summary.json` khóa ID đầu vào, quy tắc, seed, checksum và tỷ lệ thực tế.
Không ghi đè audit hoặc raw. Đọc danh sách bên trong `subjects.csv` để kiểm tra người cụ thể.

In [ ]:
SPLIT_ID = 'split_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
out = ROOT / 'dsp/data/manifests' / SPLIT_ID
report = ROOT / 'dsp/outputs/reports' / SPLIT_ID
out.mkdir(parents=True, exist_ok=False)
report.mkdir(parents=True, exist_ok=False)
for name, rows in [('events',events),('recordings',recordings),('subjects',subject_rows)]:
    write_csv(out / (name+'.csv'), rows)
write_csv(report / 'counts.csv', counts)
snapshot('02_chia_tap_theo_nguoi.ipynb', report / 'notebook_source.ipynb')
summary = dict(run_id=SPLIT_ID, audit_id=AUDIT_ID, audit_summary_sha256=sha256(audit_path),
    selected_seed=selected_seed, attempts=attempts, nominal_validation_ratio=VAL_RATIO,
    actual_subject_ratio=len(val_subjects)/len(subjects), counts=counts,
    rule='Sorted subject IDs; RandomState permutation; ceil(ratio*n) validation first; first feasible seed',
    test_audio_read=False, subject_intersection_count=0, cross_core_waveform_duplicates=0)
save_summary(report / 'split_summary.json', summary, list(out.glob('*.csv')) + list(report.glob('*.csv')) + [report/'notebook_source.ipynb'])
(report/'BAO_CAO.md').write_text('# Split đã freeze\n\nSeed: '+str(selected_seed)+
    '\n\nValidation subjects: '+str(len(val_subjects))+'/'+str(len(subjects))+
    '. Giao người/core duplicate = 0. Test chỉ metadata. Xem counts.csv và subjects.csv.\n',encoding='utf-8')
print('SPLIT_ID =', repr(SPLIT_ID))